# الفصل 18: مصادر البيانات وتنظيفها

> طبقة الحدس لو سألت أي باحث عمل فعليًا على تدريب LLM كبير: "ما الذي استغرق أكبر جزء من الوقت والجهد؟" فالجواب في الغالب ليس المعمارية ولا حتى التدريب نفسه، بل البيانات.

- **الجزء:** 5 — بناء LLM من الصفر
- **الفصل:** 18
- **المستوى:** متقدم
- **زمن القراءة:** 30 دقيقة
- **المتطلبات:** الفصل 17: المكوّنات الحديثة: RMSNorm وSwiGLU
- **الوسوم:** Common Crawl، Deduplication، Quality Filtering، PII

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch18-data-pipeline.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** beautifulsoup4, fasttext-wheel, numpy, trafilatura
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("bs4", "beautifulsoup4"),
    ("fasttext", "fasttext-wheel"),
    ("numpy", "numpy"),
    ("trafilatura", "trafilatura")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch18-data-pipeline.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch18-data-pipeline.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس لو سألت أي باحث عمل فعليًا على تدريب LLM كبير: "ما الذي استغرق أكبر جزء من الوقت والجهد؟" فالجواب في الغالب ليس المعمارية ولا حتى التدريب نفسه، بل البيانات.

> **بيانات الفصل**
>
> - **الجزء:** 5 — بناء LLM من الصفر
> - **المستوى:** متقدم
> - **زمن القراءة:** 30 دقيقة
> - **المتطلبات:** الفصل 17: المكوّنات الحديثة: RMSNorm وSwiGLU


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم مصادر البيانات الشائعة
- فهم الاستخلاص من HTML
- فهم فلترة اللغة


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**

لو سألت أي باحث عمل فعليًا على تدريب LLM كبير: "ما الذي استغرق أكبر جزء من الوقت والجهد؟" فالجواب في الغالب ليس المعمارية ولا حتى التدريب نفسه، بل **البيانات**. الجملة الشهيرة في هذا المجال:

> "Garbage in, garbage out" — لو دربت نموذجًا بتريليون رمز من نصوص رديئة ومكررة ومليئة بالضجيج، فستحصل على نموذج رديء بغض النظر عن مدى تطور معماريته.

الأرقام توضح حجم التحدي: نموذج مثل LLaMA-3 دُرِّب على **15 تريليون رمز** (Token)، ونموذج GPT-4 يُقدَّر أنه دُرِّب على كمية مشابهة أو أكبر. لكن الإنترنت الخام (Common Crawl مثلاً) يحتوي على مئات التريليونات من الرموز، وأغلبها:
- محتوى مكرر (نفس المقال يظهر في آلاف المواقع)
- محتوى منخفض الجودة (سبام، قوائم كلمات مفتاحية، محتوى آلي الإنتاج)
- معلومات شخصية حساسة (PII: أرقام هواتف، بريد إلكتروني، أرقام بطاقات)
- محتوى سام أو ضار
- تلوث بمجموعات الاختبار (Test Contamination) التي ستُستخدم لاحقًا لتقييم النموذج

مهمة هذا الفصل: بناء خط إنتاج كامل (Data Pipeline) يحوّل الإنترنت الخام الفوضوي إلى مجموعة بيانات تدريب نظيفة وعالية الجودة.


## 2. الخريطة الذهنية


```text
خط إنتاج بيانات التدريب (Data Pipeline)
│
├── 1. الجمع (Collection)
│     Common Crawl, Wikipedia, Books, GitHub Code,
│     Academic Papers (arXiv), Forums (Reddit, StackExchange)
│
├── 2. الاستخلاص (Extraction)
│     HTML → Text نظيف (إزالة الوسوم، القوائم الجانبية، الإعلانات)
│
├── 3. فلترة اللغة (Language Filtering)
│     fastText language ID → الاحتفاظ بالعربية/الإنجليزية فقط مثلاً
│
├── 4. إزالة التكرار (Deduplication)
│     - Exact Dedup (Hashing)
│     - Fuzzy Dedup (MinHash + LSH)
│
├── 5. فلترة الجودة (Quality Filtering)
│     - قواعد heuristic (طول الجملة، نسبة الأحرف الخاصة...)
│     - نموذج تصنيف جودة (Classifier-based, مثل FineWeb-Edu)
│     - فلترة Perplexity بنموذج لغوي صغير
│
├── 6. إزالة المعلومات الشخصية (PII Removal)
│     Regex + NER للكشف عن الهواتف/الإيميلات/الأرقام الوطنية
│
├── 7. فحص السمّية (Toxicity/Safety Filtering)
│
└── 8. فحص التلوث (Contamination Check)
      التأكد أن بيانات التدريب لا تحتوي على أسئلة اختبارات
      MMLU/HumanEval/GSM8K حرفيًا
```


## 3. الشرح المتدرج

### 3.1 مصادر البيانات الشائعة

> 🟡 **طبقة الآلية**

| المصدر | الحجم التقريبي | الجودة | الاستخدام النموذجي |
|---|---|---|---|
| **Common Crawl** | بتابايتات (Petabytes) | منخفضة-متوسطة (خام) | الأساس الكمي لمعظم LLMs |
| **Wikipedia** | ~20 مليار رمز (كل اللغات) | عالية جدًا | معرفة موسوعية موثوقة |
| **Books (Books3, Gutenberg)** | مئات المليارات | عالية | سرد طويل، أسلوب أدبي |
| **كود GitHub** | تريليونات الرموز | متفاوتة | قدرات البرمجة |
| **arXiv/PubMed** | عشرات المليارات | عالية جدًا (تقنية) | الاستدلال العلمي |
| **Reddit/StackExchange** | مئات المليارات | متوسطة-عالية | حوار، أسئلة وأجوبة |
| **FineWeb / RefinedWeb** | تريليونات (مُنقّاة من CC) | عالية (بعد تنقية صارمة) | الغالبية العظمى من تدريب LLMs الحديثة |
| **مصادر عربية متخصصة** | OSCAR-Arabic, Arabic Wikipedia, ArabicText 2022 | متفاوتة، تحتاج تنقية إضافية | تعزيز القدرة العربية |

> ⚠️ **خطأ شائع**: الاعتقاد بأن "المزيد من البيانات = دائمًا أفضل". الحقيقة (كما سنرى في فصل Chinchilla) أن **جودة** البيانات ونسبتها لحجم النموذج أهم من الكمية المطلقة. مليار رمز نظيف قد يتفوق على 10 مليارات رمز مليئة بالتكرار والسبام.

### 3.2 الاستخلاص من HTML

البيانات الخام من Common Crawl تأتي بصيغة WARC (صفحات HTML كاملة). يجب استخلاص النص الفعلي وتجاهل كل ما هو تنسيق:


In [ ]:
import re
from bs4 import BeautifulSoup
import trafilatura  # مكتبة متخصصة في استخلاص المحتوى الرئيسي من HTML

def extract_clean_text_from_html(html_content: str) -> str:
    """
    استخلاص النص الرئيسي من صفحة HTML مع تجاهل:
    - القوائم الجانبية (navigation, sidebar)
    - الإعلانات
    - التذييل والترويسة (footer, header)
    - الوسوم البرمجية (script, style)

    نستخدم trafilatura لأنها مصممة خصيصًا لهذه المهمة وتتفوق
    على BeautifulSoup البسيط في تمييز "المحتوى الرئيسي" عن الضوضاء.
    """
    # الطريقة المتقدمة: trafilatura (تُستخدم فعليًا في بناء مجموعات مثل RefinedWeb)
    extracted = trafilatura.extract(
        html_content,
        include_comments=False,
        include_tables=True,
        no_fallback=False,
    )
    if extracted:
        return extracted

    # طريقة احتياطية بسيطة بـ BeautifulSoup
    soup = BeautifulSoup(html_content, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    # تنظيف الأسطر الفارغة المتكررة
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(line for line in lines if line)
    return text

def basic_arabic_text_cleaning(text: str) -> str:
    """
    تنظيف أساسي خاص بالنصوص العربية:
    - توحيد أشكال الألف (أ، إ، آ → ا) [اختياري، يُستخدم أحيانًا للتدريب لكن يُفقِد معلومة]
    - إزالة التشكيل الزائد إذا لم يكن مطلوبًا
    - إزالة التكرار المفرط للأحرف (مثال: "جميلللللل" → "جميل")
    - توحيد المسافات المتعددة
    """
    # إزالة التشكيل (التنوين والحركات) - اختياري حسب حالة الاستخدام
    arabic_diacritics = re.compile(r'[\u064B-\u0652\u0670\u0640]')
    text = arabic_diacritics.sub('', text)

    # تقليص تكرار الأحرف الزائد عن 3 مرات متتالية إلى مرتين
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # توحيد المسافات
    text = re.sub(r'\s+', ' ', text).strip()

    # إزالة الأسطر القصيرة جدًا (أقل من 3 كلمات) التي غالبًا قوائم تنقل أو إعلانات
    lines = text.split('\n')
    lines = [l for l in lines if len(l.split()) >= 3]
    return '\n'.join(lines)


### 3.3 فلترة اللغة

في مجموعة بيانات متعددة اللغات مثل Common Crawl، يجب أولاً فصل النصوص حسب اللغة:


In [ ]:
import fasttext

class LanguageFilter:
    """
    فلترة النصوص حسب اللغة باستخدام نموذج fastText المدرَّب مسبقًا
    (lid.176.bin) القادر على تمييز 176 لغة بدقة عالية وسرعة كبيرة
    (مناسب لمعالجة تريليونات الرموز).
    """
    def __init__(self, model_path: str = "lid.176.bin",
                 target_langs: list = None,
                 confidence_threshold: float = 0.65):
        self.model = fasttext.load_model(model_path)
        self.target_langs = set(target_langs or ["ar", "en"])
        self.confidence_threshold = confidence_threshold

    def predict_language(self, text: str) -> tuple:
        """يعيد (رمز اللغة، درجة الثقة)"""
        # fastText يتوقع نصًا بدون أسطر جديدة
        clean_text = text.replace("\n", " ")[:2000]  # عيّنة كافية للتنبؤ
        predictions = self.model.predict(clean_text, k=1)
        lang_code = predictions[0][0].replace("__label__", "")
        confidence = predictions[1][0]
        return lang_code, confidence

    def should_keep(self, text: str) -> bool:
        lang, confidence = self.predict_language(text)
        return lang in self.target_langs and confidence >= self.confidence_threshold

    def filter_corpus(self, documents: list) -> list:
        kept = [doc for doc in documents if self.should_keep(doc)]
        print(f"احتُفظ بـ {len(kept)}/{len(documents)} "
              f"({100*len(kept)/max(len(documents),1):.1f}%) من الوثائق")
        return kept


### 3.4 إزالة التكرار (Deduplication) — القلب التقني للفصل

> 🔴 **طبقة التعمّق**

التكرار في بيانات الويب مشكلة ضخمة: نفس المقالة الإخبارية قد تُنسَخ على مئات المواقع، نفس صفحة الشروط والأحكام تتكرر حرفيًا في آلاف المواقع. أبحاث مثل دراسة Lee et al. (2022) "Deduplicating Training Data Makes Language Models Better" أثبتت أن إزالة التكرار:
- تقلل حجم البيانات بنسبة 30-50% أحيانًا
- تحسّن جودة النموذج الناتج (نفس الأداء ببيانات أقل)
- تقلل احتمال "تذكّر" النموذج لمقاطع كاملة حرفيًا (مشكلة الخصوصية وحقوق النشر)

هناك نوعان من إزالة التكرار:

**أ. Exact Deduplication (تكرار تام)**: وثيقتان متطابقتان حرفيًا بنسبة 100%. تُحل بسهولة عبر حساب Hash (مثل SHA-256) لكل وثيقة وتجميع الوثائق التي لها نفس القيمة.


In [ ]:
import hashlib
from collections import defaultdict

def exact_deduplication(documents: list) -> list:
    """
    إزالة التكرار التام: أي وثيقتين متطابقتين حرفيًا يُحتفظ بواحدة فقط منهما.
    نستخدم SHA-256 لأنه مقاوم للتصادم (Collision-resistant) بشكل عملي.
    """
    seen_hashes = set()
    unique_documents = []

    for doc in documents:
        # تطبيع بسيط قبل الهاش: إزالة المسافات الزائدة لتفادي اعتبار
        # وثيقتين متطابقتين فعليًا مختلفتين بسبب فراغ زائد
        normalized = " ".join(doc.split())
        doc_hash = hashlib.sha256(normalized.encode("utf-8")).hexdigest()

        if doc_hash not in seen_hashes:
            seen_hashes.add(doc_hash)
            unique_documents.append(doc)

    removed = len(documents) - len(unique_documents)
    print(f"Exact Dedup: أُزيلت {removed} وثيقة مكررة تمامًا "
          f"({100*removed/max(len(documents),1):.1f}%)")
    return unique_documents


**ب. Fuzzy Deduplication (تكرار تقريبي) عبر MinHash + LSH**: المشكلة الأصعب هي الوثائق **المتشابهة جدًا لكن ليست متطابقة حرفيًا** (مثال: نفس المقال مع فقرة إعلانية مختلفة، أو نسخة مُعاد صياغتها قليلاً). هنا نحتاج قياس **تشابه Jaccard** بين مجموعات الكلمات (n-grams) لكل وثيقة:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

لكن حساب التقاطع والاتحاد لكل زوج من ملايين الوثائق مكلف جدًا $O(n^2)$. الحل هو **MinHash**: تقنية توقيع احتمالي (Probabilistic Signature) تقرّب تشابه Jaccard بدقة عالية وبتكلفة أقل بكثير، مدمجة مع **LSH (Locality-Sensitive Hashing)** لتجميع الوثائق المتشابهة في "دلاء" (Buckets) دون مقارنة كل الأزواج.

**حدس MinHash**: لو طبّقنا دالة هاش عشوائية على كل عنصر في مجموعة، وأخذنا الحد الأدنى (Min) من قيم الهاش، فإن احتمال تطابق الحد الأدنى بين مجموعتين يساوي بالضبط تشابه Jaccard بينهما:

$$P(\min(h(A)) = \min(h(B))) = J(A, B)$$

بتكرار هذه العملية بعدة دوال هاش مختلفة (مثلاً 128 دالة) والمتوسط، نحصل على تقدير دقيق لـ J(A,B) دون الحاجة لحساب التقاطع/الاتحاد الفعلي.


In [ ]:
import numpy as np
from collections import defaultdict
import hashlib

class MinHashDeduplicator:
    """
    إزالة التكرار التقريبي باستخدام MinHash + LSH.
    هذه هي التقنية الفعلية المستخدمة في بناء مجموعات مثل
    RefinedWeb و FineWeb لإزالة مليارات الوثائق شبه المكررة
    بكفاءة زمنية تقارب O(n) بدلاً من O(n²).
    """
    def __init__(self, num_hashes: int = 128, num_bands: int = 16,
                 ngram_size: int = 5, similarity_threshold: float = 0.8):
        self.num_hashes = num_hashes
        self.num_bands = num_bands
        self.rows_per_band = num_hashes // num_bands
        self.ngram_size = ngram_size
        self.similarity_threshold = similarity_threshold

        # توليد معاملات دوال الهاش العشوائية (a، b لكل دالة هاش خطية)
        # نستخدم صيغة: h_i(x) = (a_i * x + b_i) mod p
        rng = np.random.RandomState(42)
        self.max_prime = (1 << 61) - 1  # عدد أولي كبير (Mersenne Prime)
        self.a_params = rng.randint(1, self.max_prime, size=num_hashes)
        self.b_params = rng.randint(0, self.max_prime, size=num_hashes)

    def _get_ngrams(self, text: str) -> set:
        """تحويل النص إلى مجموعة n-grams على مستوى الكلمات"""
        words = text.split()
        if len(words) < self.ngram_size:
            return {tuple(words)}
        return {tuple(words[i:i + self.ngram_size])
                for i in range(len(words) - self.ngram_size + 1)}

    def _hash_ngram(self, ngram: tuple) -> int:
        """تحويل n-gram إلى عدد صحيح عبر MD5 (أسرع من SHA للأغراض الداخلية)"""
        ngram_str = " ".join(ngram)
        return int(hashlib.md5(ngram_str.encode("utf-8")).hexdigest(), 16)

    def compute_minhash_signature(self, text: str) -> np.ndarray:
        """
        حساب توقيع MinHash لوثيقة: مصفوفة من num_hashes قيمة،
        كل قيمة هي أصغر قيمة هاش بين كل الـ n-grams للوثيقة
        تحت دالة هاش معينة.
        """
        ngrams = self._get_ngrams(text)
        if not ngrams:
            return np.full(self.num_hashes, self.max_prime, dtype=np.int64)

        ngram_hashes = np.array([self._hash_ngram(ng) for ng in ngrams])
        # تطبيق كل دالة هاش خطية على كل قيم الـ ngram_hashes دفعة واحدة
        signature = np.full(self.num_hashes, self.max_prime, dtype=np.int64)
        for i in range(self.num_hashes):
            hashed_values = (self.a_params[i] * ngram_hashes +
                              self.b_params[i]) % self.max_prime
            signature[i] = hashed_values.min()
        return signature

    def estimate_jaccard_similarity(self, sig1: np.ndarray, sig2: np.ndarray) -> float:
        """تقدير تشابه Jaccard من نسبة تطابق عناصر التوقيعين"""
        return float(np.mean(sig1 == sig2))

    def build_lsh_buckets(self, signatures: list) -> dict:
        """
        تقنية LSH: نقسّم كل توقيع إلى (num_bands) نطاقات (Bands)،
        كل نطاق يحتوي (rows_per_band) قيمة. وثيقتان تقعان في نفس
        الدلو (Bucket) لأي نطاق تصبحان "مرشحتين" للمقارنة الدقيقة.
        هذا يقلل عدد المقارنات من O(n²) إلى تقريبًا O(n).
        """
        buckets = defaultdict(list)
        for doc_idx, signature in enumerate(signatures):
            for band_idx in range(self.num_bands):
                start = band_idx * self.rows_per_band
                end = start + self.rows_per_band
                band_signature = tuple(signature[start:end])
                bucket_key = (band_idx, hash(band_signature))
                buckets[bucket_key].append(doc_idx)
        return buckets

    def deduplicate(self, documents: list) -> list:
        """خط الإنتاج الكامل: حساب التواقيع → بناء الدلاء → تصفية المكررات"""
        print(f"حساب تواقيع MinHash لـ {len(documents)} وثيقة...")
        signatures = [self.compute_minhash_signature(doc) for doc in documents]

        print("بناء دلاء LSH...")
        buckets = self.build_lsh_buckets(signatures)

        # تجميع أزواج الوثائق المرشحة للمقارنة (ظهرت معًا في دلو واحد على الأقل)
        candidate_pairs = set()
        for bucket_docs in buckets.values():
            if len(bucket_docs) > 1:
                for i in range(len(bucket_docs)):
                    for j in range(i + 1, len(bucket_docs)):
                        pair = tuple(sorted([bucket_docs[i], bucket_docs[j]]))
                        candidate_pairs.add(pair)

        print(f"عدد الأزواج المرشحة للمقارنة: {len(candidate_pairs)} "
              f"(بدلاً من {len(documents)*(len(documents)-1)//2} في الطريقة الساذجة)")

        # تحديد الوثائق المكررة فعليًا عبر حساب التشابه الدقيق للمرشحين فقط
        to_remove = set()
        for i, j in candidate_pairs:
            if i in to_remove or j in to_remove:
                continue
            similarity = self.estimate_jaccard_similarity(signatures[i], signatures[j])
            if similarity >= self.similarity_threshold:
                # نحتفظ بالوثيقة الأطول عادة (غالبًا الأصلية الكاملة)
                if len(documents[i]) >= len(documents[j]):
                    to_remove.add(j)
                else:
                    to_remove.add(i)

        unique_documents = [doc for idx, doc in enumerate(documents)
                             if idx not in to_remove]
        print(f"Fuzzy Dedup: أُزيلت {len(to_remove)} وثيقة شبه مكررة "
              f"({100*len(to_remove)/max(len(documents),1):.1f}%)")
        return unique_documents


> 💡 **نصيحة عملية**: في الأنظمة الصناعية الحقيقية (مثل بناء RefinedWeb أو FineWeb)، لا تُستخدم هذه الخوارزميات على عقدة واحدة؛ بل تُنفَّذ موزَّعة عبر Apache Spark أو أنظمة مشابهة على آلاف العقد، لأن حجم البيانات (تريليونات الوثائق) يتجاوز قدرة أي جهاز واحد. لكن المبدأ الرياضي (MinHash + LSH) يبقى كما هو تمامًا.

### 3.5 فلترة الجودة (Quality Filtering)

هناك ثلاث مقاربات رئيسية لفلترة الجودة، تُستخدم عادة معًا بشكل متتابع:

**أ. مرشّحات Heuristic (قواعد يدوية)**: مستوحاة من ورقة Gopher/MassiveText ومستخدمة في معظم خطوط الإنتاج الحديثة:


In [ ]:
class HeuristicQualityFilter:
    """
    مجموعة قواعد بسيطة لكن فعّالة جدًا لفلترة النصوص منخفضة الجودة،
    مستوحاة من أوراق Gopher وC4 وFineWeb. هذه القواعد رخيصة حسابيًا
    وتُطبَّق كخط دفاع أول قبل المرشّحات الأثقل (نماذج التصنيف).
    """
    def __init__(self,
                 min_words: int = 50,
                 max_words: int = 100_000,
                 min_avg_word_length: float = 2.0,
                 max_avg_word_length: float = 15.0,
                 max_symbol_to_word_ratio: float = 0.3,
                 max_repeated_line_fraction: float = 0.3):
        self.min_words = min_words
        self.max_words = max_words
        self.min_avg_word_length = min_avg_word_length
        self.max_avg_word_length = max_avg_word_length
        self.max_symbol_to_word_ratio = max_symbol_to_word_ratio
        self.max_repeated_line_fraction = max_repeated_line_fraction

    def check_document(self, text: str) -> tuple:
        """
        يعيد (يُقبَل: bool, سبب الرفض إن وُجد: str)
        نرتب الفحوصات من الأرخص للأغلى حسابيًا (fail-fast).
        """
        words = text.split()
        num_words = len(words)

        # 1. فحص طول الوثيقة
        if num_words < self.min_words:
            return False, f"قصيرة جدًا ({num_words} كلمة)"
        if num_words > self.max_words:
            return False, f"طويلة جدًا ({num_words} كلمة)"

        # 2. متوسط طول الكلمة (كلمات قصيرة جدًا = سبام/قوائم، طويلة جدًا = كود ملتصق/أخطاء)
        avg_word_len = sum(len(w) for w in words) / num_words
        if not (self.min_avg_word_length <= avg_word_len <= self.max_avg_word_length):
            return False, f"متوسط طول الكلمة غير طبيعي ({avg_word_len:.1f})"

        # 3. نسبة الرموز غير الأبجدية (علامات ترقيم، رموز خاصة) إلى الكلمات
        num_symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
        symbol_ratio = num_symbols / max(num_words, 1)
        if symbol_ratio > self.max_symbol_to_word_ratio:
            return False, f"نسبة رموز مرتفعة ({symbol_ratio:.2f})"

        # 4. نسبة الأسطر المكررة داخل نفس الوثيقة (مؤشر على قوائم/قوالب متكررة)
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if lines:
            unique_lines = len(set(lines))
            repeated_fraction = 1 - (unique_lines / len(lines))
            if repeated_fraction > self.max_repeated_line_fraction:
                return False, f"نسبة تكرار أسطر مرتفعة ({repeated_fraction:.2f})"

        # 5. فحص وجود كلمة واحدة على الأقل تحتوي حرفًا أبجديًا (رفض الوثائق الرقمية بحتة)
        has_alpha_word = any(any(c.isalpha() for c in w) for w in words)
        if not has_alpha_word:
            return False, "لا تحتوي كلمات نصية فعلية"

        return True, None

    def filter_corpus(self, documents: list) -> tuple:
        kept, rejected_reasons = [], defaultdict(int)
        for doc in documents:
            accepted, reason = self.check_document(doc)
            if accepted:
                kept.append(doc)
            else:
                rejected_reasons[reason.split("(")[0].strip()] += 1

        print(f"Heuristic Filter: احتُفظ بـ {len(kept)}/{len(documents)}")
        for reason, count in sorted(rejected_reasons.items(),
                                     key=lambda x: -x[1]):
            print(f"   - رُفضت {count} وثيقة بسبب: {reason}")
        return kept, rejected_reasons


**ب. فلترة بنموذج تصنيف الجودة (Classifier-based)**: تُدرَّب شبكة تصنيف صغيرة (مثل FastText أو نموذج Transformer صغير) على التمييز بين عيّنات "عالية الجودة" (مثل Wikipedia والكتب المنشورة) و"منخفضة الجودة" (عينة عشوائية من الويب الخام)، ثم تُستخدم لتسجيل كل وثيقة بدرجة جودة والاحتفاظ بأعلى نسبة مئوية فقط. هذا هو أساس منهجية **FineWeb-Edu** التي حققت قفزة نوعية في جودة بيانات التدريب.


In [ ]:
import fasttext

class QualityClassifierFilter:
    """
    نموذج فلترة جودة مبني على fastText، يُحاكي منهجية FineWeb-Edu:
    يُدرَّب على تمييز النصوص "التعليمية عالية القيمة" من بقية نصوص الويب.
    """
    def __init__(self, positive_samples: list, negative_samples: list):
        """
        positive_samples: نصوص عالية الجودة (ويكيبيديا، كتب، مقالات أكاديمية)
        negative_samples: عينة عشوائية من الويب الخام (غير مفلترة)
        """
        self.training_file = "_quality_classifier_train.txt"
        self._prepare_training_data(positive_samples, negative_samples)
        self.model = fasttext.train_supervised(
            input=self.training_file,
            lr=0.5, epoch=25, wordNgrams=2, dim=100,
        )

    def _prepare_training_data(self, positive, negative):
        with open(self.training_file, "w", encoding="utf-8") as f:
            for text in positive:
                clean = text.replace("\n", " ")
                f.write(f"__label__high_quality {clean}\n")
            for text in negative:
                clean = text.replace("\n", " ")
                f.write(f"__label__low_quality {clean}\n")

    def score_document(self, text: str) -> float:
        """يعيد درجة بين 0 و1، حيث 1 تعني ثقة كاملة بأنها عالية الجودة"""
        clean_text = text.replace("\n", " ")[:2000]
        labels, probs = self.model.predict(clean_text, k=1)
        label = labels[0].replace("__label__", "")
        prob = probs[0]
        return prob if label == "high_quality" else (1 - prob)

    def filter_corpus(self, documents: list, keep_top_fraction: float = 0.5) -> list:
        """الاحتفاظ بأعلى نسبة مئوية من الوثائق حسب درجة الجودة"""
        scored = [(doc, self.score_document(doc)) for doc in documents]
        scored.sort(key=lambda x: -x[1])
        num_to_keep = int(len(scored) * keep_top_fraction)
        kept = [doc for doc, score in scored[:num_to_keep]]
        print(f"Quality Classifier: احتُفظ بأعلى {keep_top_fraction*100:.0f}% "
              f"= {len(kept)} وثيقة")
        return kept


**ج. فلترة Perplexity**: تُستخدم نموذج لغوي صغير مدرَّب على بيانات نظيفة (مثل Wikipedia) لحساب حيرة (Perplexity) كل وثيقة مرشَّحة. الوثائق التي تحصل على Perplexity عالٍ جدًا (غير متوقَّعة لغويًا — غالبًا سبام أو نص مشوَّه) تُرفَض.

### 3.6 إزالة المعلومات الشخصية (PII Removal)


In [ ]:
import re

class PIIRemover:
    """
    إزالة أو إخفاء المعلومات الشخصية القابلة للتحديد (PII) من النصوص
    قبل التدريب، للامتثال للخصوصية ومنع النموذج من "حفظ" بيانات حساسة.
    نستخدم أنماط Regex للحالات الواضحة، ويمكن تعزيزها بنموذج NER
    (Named Entity Recognition) للكشف عن الأسماء الشخصية والعناوين.
    """
    def __init__(self):
        self.patterns = {
            "EMAIL": re.compile(
                r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
            ),
            "PHONE_INTL": re.compile(
                r'\+?\d{1,3}[-.\s]?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}'
            ),
            "SSN_US": re.compile(r'\b\d{3}-\d{2}-\d{4}\b'),
            "CREDIT_CARD": re.compile(
                r'\b(?:\d[ -]*?){13,16}\b'
            ),
            "IP_ADDRESS": re.compile(
                r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
            ),
            "SAUDI_NATIONAL_ID": re.compile(r'\b[12]\d{9}\b'),
        }
        self.replacement_tokens = {
            "EMAIL": "[EMAIL]",
            "PHONE_INTL": "[PHONE]",
            "SSN_US": "[SSN]",
            "CREDIT_CARD": "[CREDIT_CARD]",
            "IP_ADDRESS": "[IP_ADDRESS]",
            "SAUDI_NATIONAL_ID": "[NATIONAL_ID]",
        }

    def scrub_text(self, text: str) -> tuple:
        """يعيد (النص المنظف، عدد كل نوع تم إخفاؤه)"""
        counts = defaultdict(int)
        cleaned_text = text
        for pii_type, pattern in self.patterns.items():
            matches = pattern.findall(cleaned_text)
            counts[pii_type] = len(matches)
            cleaned_text = pattern.sub(self.replacement_tokens[pii_type],
                                         cleaned_text)
        return cleaned_text, dict(counts)

    def scrub_corpus(self, documents: list) -> list:
        total_counts = defaultdict(int)
        cleaned_docs = []
        for doc in documents:
            cleaned, counts = self.scrub_text(doc)
            cleaned_docs.append(cleaned)
            for k, v in counts.items():
                total_counts[k] += v
        print("إحصائيات إزالة PII عبر الكوربص بالكامل:")
        for pii_type, count in total_counts.items():
            if count > 0:
                print(f"   - {pii_type}: {count} حالة")
        return cleaned_docs


### 3.7 فحص التلوث (Contamination Check)

> ⚠️ **خطأ شائع خطير**: إذا احتوت بيانات التدريب على أسئلة اختبارات معيارية مثل MMLU أو GSM8K أو HumanEval **حرفيًا** (لأنها موجودة على الإنترنت ونُسخت ضمن Common Crawl)، فإن النموذج "يحفظ" الإجابات بدلاً من "تعلّم" حل المشكلة، مما يُنتج أرقام أداء مُضلِّلة وغير قابلة للتعميم. هذه المشكلة سُمِّيت **Test Set Contamination** وتم رصدها فعليًا في عدة نماذج مفتوحة المصدر عند الفحص اللاحق.


In [ ]:
class ContaminationChecker:
    """
    يفحص ما إذا كانت وثائق التدريب تحتوي على مقاطع متطابقة أو شبه متطابقة
    من مجموعات بيانات التقييم المعروفة (benchmark test sets)، باستخدام
    تطابق n-grams طويلة (13-gram فأكثر) وهي الطريقة المعتمدة في تقارير
    GPT-4 وLLaMA الفنية.
    """
    def __init__(self, benchmark_texts: list, ngram_size: int = 13):
        self.ngram_size = ngram_size
        self.benchmark_ngrams = set()
        for text in benchmark_texts:
            self.benchmark_ngrams.update(self._get_ngrams(text))

    def _get_ngrams(self, text: str) -> set:
        words = text.lower().split()
        if len(words) < self.ngram_size:
            return {tuple(words)}
        return {tuple(words[i:i + self.ngram_size])
                for i in range(len(words) - self.ngram_size + 1)}

    def is_contaminated(self, document: str, overlap_threshold: int = 1) -> bool:
        """يعيد True إذا احتوت الوثيقة على عدد n-grams مشتركة مع benchmark
        يساوي أو يتجاوز overlap_threshold"""
        doc_ngrams = self._get_ngrams(document)
        overlap = doc_ngrams & self.benchmark_ngrams
        return len(overlap) >= overlap_threshold

    def filter_corpus(self, documents: list) -> list:
        clean_docs = [doc for doc in documents if not self.is_contaminated(doc)]
        removed = len(documents) - len(clean_docs)
        print(f"Contamination Check: أُزيلت {removed} وثيقة ملوَّثة "
              f"من مجموعات التقييم")
        return clean_docs


## 4. مثال محلول خطوة بخطوة

لنطبّق خط الإنتاج الكامل على مجموعة صغيرة من 6 وثائق عربية توضيحية:


In [ ]:
raw_documents = [
    # وثيقة 1: نص جيد عن الذكاء الاصطناعي
    "الذكاء الاصطناعي هو مجال علمي يهتم بتطوير أنظمة قادرة على أداء "
    "مهام تتطلب عادة ذكاءً بشريًا مثل الفهم والتعلم واتخاذ القرار. "
    "شهد هذا المجال تطورًا هائلاً في العقد الأخير بفضل التعلم العميق.",

    # وثيقة 2: تكرار شبه تام للوثيقة 1 (تغيير كلمة واحدة فقط)
    "الذكاء الاصطناعي هو مجال علمي يهتم بتطوير أنظمة قادرة على أداء "
    "مهام تتطلب عادة ذكاءً بشريًا مثل الفهم والتعلم واتخاذ القرارات. "
    "شهد هذا المجال تطورًا هائلاً في العقد الأخير بفضل التعلم العميق.",

    # وثيقة 3: نص قصير جدًا (سيُرفَض بفلتر الطول)
    "مرحبا بك",

    # وثيقة 4: يحتوي PII (بريد إلكتروني ورقم هاتف)
    "للتواصل معنا يرجى إرسال بريد إلكتروني إلى ahmed.sales@example.com "
    "أو الاتصال على الرقم +966-50-123-4567 خلال ساعات الدوام الرسمي من "
    "الأحد إلى الخميس للاستفسار عن خدماتنا المتنوعة في مجال البرمجيات.",

    # وثيقة 5: سبام/قائمة كلمات مفتاحية (سيُرفَض بفلتر نسبة الرموز)
    "!!!خصم خصم خصم!!! اشتري الآن ### عرض محدود $$$ سارع بالطلب !!! "
    "### اتصل الآن $$$ !!! خصم 90% ### !!!",

    # وثيقة 6: نص جيد ومختلف تمامًا
    "تُعد معالجة اللغات الطبيعية أحد أهم فروع الذكاء الاصطناعي، وتتضمن "
    "مهامًا متنوعة مثل الترجمة الآلية والتلخيص والإجابة على الأسئلة. "
    "ساهمت نماذج المحوّلات في إحداث نقلة نوعية في جودة هذه التطبيقات.",
]

print("=" * 60)
print(f"البدء بـ {len(raw_documents)} وثيقة خام")
print("=" * 60)

# الخطوة 1: فلترة الجودة Heuristic (يُزيل الوثيقة 3 والوثيقة 5)
quality_filter = HeuristicQualityFilter(min_words=5, max_symbol_to_word_ratio=0.3)
step1_docs, _ = quality_filter.filter_corpus(raw_documents)
# → يبقى: [1, 2, 4, 6]

# الخطوة 2: إزالة PII (يُنظّف الوثيقة 4 دون حذفها)
pii_remover = PIIRemover()
step2_docs = pii_remover.scrub_corpus(step1_docs)
# → الوثيقة 4 أصبحت: "... [EMAIL] ... [PHONE] ..."

# الخطوة 3: Fuzzy Deduplication (يُزيل إحدى الوثيقتين 1/2 لتشابههما الشديد)
deduplicator = MinHashDeduplicator(num_hashes=64, num_bands=8,
                                     ngram_size=3, similarity_threshold=0.7)
step3_docs = deduplicator.deduplicate(step2_docs)
# → يبقى 3 وثائق نهائية: [وثيقة 1 أو 2، الوثيقة 4 منظفة، الوثيقة 6]

print("\n" + "=" * 60)
print(f"النتيجة النهائية: {len(step3_docs)} وثيقة نظيفة من أصل {len(raw_documents)}")
print("=" * 60)
for i, doc in enumerate(step3_docs, 1):
    print(f"\n[{i}] {doc[:100]}...")


**الناتج المتوقع** (تفصيليًا):


```text
============================================================
البدء بـ 6 وثيقة خام
============================================================
Heuristic Filter: احتُفظ بـ 4/6
   - رُفضت 1 وثيقة بسبب: قصيرة جدًا
   - رُفضت 1 وثيقة بسبب: نسبة رموز مرتفعة

إحصائيات إزالة PII عبر الكوربص بالكامل:
   - EMAIL: 1 حالة
   - PHONE_INTL: 1 حالة

حساب تواقيع MinHash لـ 4 وثيقة...
بناء دلاء LSH...
عدد الأزواج المرشحة للمقارنة: 1
Fuzzy Dedup: أُزيلت 1 وثيقة شبه مكررة (25.0%)

============================================================
النتيجة النهائية: 3 وثيقة نظيفة من أصل 6
============================================================
```


لاحظ كيف تدرّجت العملية: من 6 → 4 (heuristic) → تنظيف PII دون تغيير العدد → 3 (dedup). هذا بالضبط نفس التدرّج الذي يحدث على نطاق تريليونات الوثائق في بناء مجموعات مثل FineWeb، لكن بأدوات موزَّعة أضخم.


## 5. نصائح وحيل

- **رتّب الفلاتر من الأرخص للأغلى حسابيًا**: ابدأ بفلترة اللغة والطول (رخيصة جدًا)، ثم Heuristic Rules، ثم Deduplication، وأخيرًا المرشّحات الثقيلة (نماذج التصنيف، فحص التلوث). هذا يوفر حوسبة هائلة لأنك تتجنب تشغيل عمليات مكلفة على بيانات كان من الممكن استبعادها مبكرًا برخص.
- **احتفظ بنسخة من البيانات الخام قبل كل مرحلة**: لتتمكن من قياس أثر كل فلتر بمفرده (Ablation)، وتصحيح الأخطاء إذا تبيّن أن فلترًا معينًا متشدد جدًا ويحذف محتوى جيدًا.
- **استخدم عيّنة صغيرة (1% مثلاً) لضبط عتبات الفلاتر** قبل تطبيقها على كامل البيانات (تريليونات الرموز)، فتصحيح خطأ في عتبة فلتر بعد معالجة كامل البيانات مكلف جدًا من ناحية الوقت.
- بالنسبة للعربية تحديدًا: انتبه لمشكلة **التطبيع (Normalization)** المختلفة (أ/إ/آ، ة/ه، ى/ي) التي قد تُفسد خوارزميات Deduplication إذا لم تُوحَّد الأشكال أولاً قبل حساب n-grams.


## 6. أخطاء شائعة ومصائد

| الخطأ | لماذا هو خطأ | الحل |
|---|---|---|
| تجاهل Deduplication لأنها "مكلفة حسابيًا" | التكرار يُضعف جودة النموذج بشكل كبير ويُهدر حوسبة تدريب مكررة | استخدام MinHash+LSH الفعّال بدلاً من المقارنة الساذجة |
| فلترة الجودة بنموذج مدرَّب فقط على الإنجليزية | يرفض نصوصًا عربية جيدة بالخطأ لعدم فهم بنيتها اللغوية | تدريب/استخدام مُصنِّف جودة خاص باللغة العربية |
| عدم فحص التلوث مع مجموعات التقييم | أرقام تقييم مُضلِّلة توحي بأداء النموذج أفضل مما هو عليه فعليًا | فحص إلزامي بـ n-gram overlap قبل الإعلان عن أي نتيجة بنشمارك |
| حذف كل النصوص التي تحتوي أرقام هواتف/إيميلات بدلاً من إخفائها فقط | فقدان سياق لغوي مفيد (الجملة المحيطة) بلا داعٍ | إخفاء (Masking) الـ PII فقط، مع الحفاظ على باقي النص |


## 7. جدول مقارنة

| الطريقة | الدقة | السرعة | قابلية التوسع | يكتشف ماذا |
|---|---|---|---|---|
| Exact Hash (SHA-256) | تامة للتطابق الحرفي | فائقة السرعة | ممتازة (تريليونات الوثائق) | فقط النسخ الحرفية 100% |
| MinHash + LSH | تقريبية (قابلة للضبط) | سريعة جدًا | ممتازة (مُستخدمة صناعيًا) | التشابه الجزئي (>70-90%) |
| مقارنة ساذجة بكل الأزواج O(n²) | تامة | بطيئة جدًا جدًا | ضعيفة (مستحيلة فوق ملايين الوثائق) | أي درجة تشابه مُحدَّدة |
| Embedding Similarity (دلالي) | تقريبية لكن تلتقط إعادة الصياغة | متوسطة (تتطلب حوسبة Embeddings) | متوسطة | التشابه الدلالي وليس فقط النصي الحرفي |


## 8. ملخص أفضل الممارسات

- ابنِ خط الإنتاج كمراحل متتابعة قابلة للتفتيش والقياس بشكل منفصل.
- استخدم MinHash+LSH للتوسع إلى حجم الإنترنت بدلاً من المقارنة الثنائية الساذجة.
- اجمع بين فلترة Heuristic الرخيصة ومُصنِّف جودة متقدم للحصول على أفضل توازن تكلفة/جودة.
- أخفِ PII بدلاً من حذف الوثيقة بالكامل حين يكون ذلك ممكنًا.
- افحص التلوث مع **كل** مجموعات التقييم التي ستستخدمها لاحقًا، قبل بدء التدريب الفعلي وليس بعده.
- احتفظ بسجلات (logs) كاملة لكل خطوة فلترة لمعرفة أثرها الكمي على حجم وجودة البيانات.


## 9. أسئلة مقابلات

**سؤال 1**: لماذا نفضّل MinHash+LSH على حساب تشابه Jaccard الدقيق مباشرة؟

> **الإجابة النموذجية**: حساب تشابه Jaccard الدقيق بين كل زوج من n وثيقة يتطلب $O(n^2)$ مقارنة، وهو غير عملي لمليارات الوثائق (تريليونات العمليات). MinHash يقرّب هذا التشابه عبر توقيع صغير الحجم ثابت الطول (128 قيمة مثلاً) بغض النظر عن طول الوثيقة، مما يجعل المقارنة بين وثيقتين عملية سريعة $O(k)$ حيث k ثابت. LSH تذهب أبعد: بدلاً من مقارنة كل الأزواج حتى باستخدام MinHash، تُجمّع الوثائق في دلاء بناءً على تشابه توقيعاتها الجزئية، بحيث لا نقارن إلا الأزواج التي يُحتمل أن تكون متشابهة فعلاً — هذا يحوّل التعقيد الفعلي إلى شبه خطي $O(n)$ في الممارسة العملية.

**سؤال 2**: ما الفرق بين فلترة الجودة القائمة على القواعد (Heuristic) والقائمة على نموذج تصنيف (Classifier-based)؟ متى تستخدم كل منهما؟

> **الإجابة النموذجية**: الفلترة القائمة على القواعد سريعة جدًا وقابلة للتفسير الكامل (تعرف بالضبط لماذا رُفضت كل وثيقة)، لكنها محدودة في التقاط الفروقات الدقيقة في الجودة (مثلاً: نص سليم نحويًا لكن ضحل المعنى). الفلترة بنموذج تصنيف (مثل FineWeb-Edu) تلتقط إشارات أعمق كالقيمة التعليمية أو العمق الفكري للنص، لكنها أبطأ وتحتاج بيانات تدريب موسومة (إيجابية/سلبية) وتفتقر للشفافية الكاملة. المقاربة العملية الأفضل: استخدام Heuristic كمرشّح أول رخيص لإزالة الحالات الواضحة (سبام، نصوص قصيرة) ثم تطبيق المُصنِّف على ما تبقى لضبط الجودة بدقة أعلى.

**سؤال 3**: لماذا يُعدّ فحص Contamination خطوة إلزامية ولا يمكن تجاهلها عند تقييم LLM؟

> **الإجابة النموذجية**: لو احتوت بيانات التدريب على أسئلة اختبار معيارية (MMLU، GSM8K...) أو إجاباتها حرفيًا، فإن النموذج قد "يحفظ" الإجابة الصحيحة بدلاً من تطوير قدرة استدلال حقيقية، مما يُنتج نتائج تقييم مرتفعة بشكل مُضلِّل لا تعكس قدرة النموذج الفعلية على التعميم لمسائل جديدة. هذا يقوّض مصداقية أي مقارنة بين النماذج، وقد أدى تاريخيًا إلى جدل علني حول نماذج اشتُبه بتلوّث بياناتها. الحل المعياري هو فحص التداخل بـ n-grams طويلة (13 كلمة فأكثر) بين كل وثيقة تدريب وكل مجموعة تقييم مخطط استخدامها، وإزالة أي وثيقة متداخلة قبل بدء التدريب.


## 10. تمرين عملي

نفّذ خط إنتاج بيانات كامل على مجموعة من 50 "وثيقة" عربية اصطناعية تتضمن:
- 10 أزواج من الوثائق شبه المكررة (بتغييرات طفيفة)
- 5 وثائق تحتوي معلومات شخصية (بريد، هاتف)
- 5 وثائق قصيرة جدًا أو سبام
- الباقي وثائق متنوعة الجودة

المطلوب: طبّق `HeuristicQualityFilter` ثم `PIIRemover` ثم `MinHashDeduplicator` بالتسلسل، واطبع تقريرًا إحصائيًا يوضح عدد الوثائق المتبقية بعد كل مرحلة ونسبة التخفيض الكلية.


## 11. مراجع للتعمّق

- Lee et al. (2022), *"Deduplicating Training Data Makes Language Models Better"* — الورقة التي أسست لأهمية Dedup في جودة LLMs.
- Penedo et al. (2023), *"The RefinedWeb Dataset for Falcon LLM"* — منهجية بناء بيانات ويب نظيفة بالكامل من Common Crawl.
- Penedo et al. (2024), *"FineWeb: Decanting the Web for the Finest Text Data at Scale"* — بما فيها FineWeb-Edu ومُصنِّف الجودة التعليمية.
- Rae et al. (2021), *"Scaling Language Models: Methods, Analysis & Insights from Training Gopher"* — مصدر قواعد الـ Heuristic Filtering الأشهر.
- Broder (1997), *"On the resemblance and containment of documents"` — الورقة الأصلية لـ MinHash.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — بناء LLM من الصفر / الفصل 18: مصادر البيانات وتنظيفها.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
